# Layer 2 - VLM forensic report

Moi sample di het chuoi ngay trong luc chay, khong build evidence truoc:

```
row -> Layer 1 (proba + SHAP) -> RAG k-NN -> 5 anh -> VLM -> report
```

Cac cell nap tuan tu. Chay tu tren xuong.

| Cell | Viec |
|---|---|
| 1-2 | Import, kiem tra duong dan |
| 3-5 | Nap Layer 1, RAG, VLM |
| 6-7 | Lay mau 50 TP + 50 FP tu val_binary |
| 8 | **Mot sample -> mot report** (xem ky truoc khi chay ca lo) |
| 9 | Chay ca lo |
| 10-13 | Correction/retention, quet confidence, latency |
| 14 | Do do nhay voi prompt (tuy chon) |

## 0. Setup tren Kaggle

**Local:** BO QUA cell nay.

**Kaggle:** code lay tu GitHub (private), data lay tu Kaggle Dataset.

Chuan bi mot lan:

1. **Token GitHub** — github.com/settings/tokens > Generate new token (classic)
   > tick `repo` > copy
2. **Luu vao Kaggle** — trong notebook: Add-ons > Secrets > Add a new secret,
   Label = `GITHUB_TOKEN`, Value = token, Save, roi tick **Attached**
3. **Gan dataset** — panel phai > Add Input > `layer2-data`

Roi bo comment cell duoi, sua `DATA` neu ten dataset khac, chay TRUOC moi cell.

| Bien | Y nghia |
|---|---|
| `UTS_DATA` | noi co `data/` + `artifacts/` (Kaggle dataset, read-only) |
| `UTS_OUT` | noi GHI ket qua — bat buoc tro ra `/kaggle/working` |

In [ ]:
# import os, sys, subprocess
# from kaggle_secrets import UserSecretsClient
#
# TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
# REPO  = f"https://{TOKEN}@github.com/TrinhPhat2082005/UTS-URA-Toward-Zero-Shot-Capable-Network-Intrusion-Detection.git"
# CODE  = "/kaggle/working/UTS-URA"
# DATA  = "/kaggle/input/layer2-data"      # sua neu ten dataset khac
#
# if not os.path.isdir(CODE):
#     subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
# else:
#     subprocess.run(["git", "-C", CODE, "pull"], check=True)
#
# LAYER2 = os.path.join(CODE, "LAYER2")
# os.environ["UTS_DATA"] = DATA
# os.environ["UTS_OUT"]  = "/kaggle/working/out"
# os.makedirs(os.environ["UTS_OUT"], exist_ok=True)
#
# for p in (LAYER2, os.path.join(LAYER2, "models")):
#     if p not in sys.path:
#         sys.path.insert(0, p)
# os.chdir(LAYER2)
#
# print("code:", LAYER2)
# print("data:", DATA)
# print("out :", os.environ["UTS_OUT"])

### Cai dependency

Kaggle da co torch/transformers/xgboost/shap. Hai goi duoi thuong thieu hoac
qua cu - ban bitsandbytes cu bao `ImportError: requires bitsandbytes>=0.46.1`
khi nap model 4-bit.

Restart session la mat het goi cai them, nen chay lai cell nay moi lan restart.
Cai xong bitsandbytes PHAI restart kernel (no nap thu vien CUDA luc import).

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" qwen-vl-utils

### Khong chac ten dataset?

Chay cell nay de xem Kaggle mount no thanh gi.

In [ ]:
# import os
# for d in sorted(os.listdir("/kaggle/input")):
#     print(d, "->", sorted(os.listdir(f"/kaggle/input/{d}"))[:6])

### Sua code roi `git pull` lai?

Python giu module da import trong bo nho, nen phai reload sau khi pull.

In [ ]:
# import subprocess, importlib, config, prompts, pipeline, metrics
# subprocess.run(["git", "-C", "/kaggle/working/UTS-URA", "pull"], check=True)
# for m in (config, prompts, pipeline, metrics):
#     importlib.reload(m)
# config.describe()

## 1. Import

In [ ]:
import os, sys, json, time, importlib
import numpy as np
import pandas as pd

# Cho phep import cac module canh notebook
HERE = os.getcwd()
if HERE not in sys.path:
    sys.path.insert(0, HERE)

import config, prompts, pipeline, metrics
for m in (config, prompts, pipeline, metrics):
    importlib.reload(m)

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)
print("import xong")

## 2. Kiem tra cau hinh

Dung lai ngay o day neu co file thieu, thay vi hong giua chung khi da nap VLM.

In [ ]:
ok = config.describe()
assert ok, "Thieu file - sua duong dan trong config.py truoc khi chay tiep."

## 3. Layer 1

XGBoost gate + TreeExplainer. SHAP la gia tri chinh xac (khong lay mau), nen
vector nay dung lam khoa truy hoi duoc.

In [ ]:
l1 = pipeline.Layer1Engine(variant=config.VARIANT)

## 4. RAG - SHAP k-NN

Khop vector SHAP cua flow voi TOAN BO build pool cua tung label (cosine lon
nhat), khong phai voi mot centroid trung binh.

`Infilteration` bi loai khoi pool vi no la lop zero-shot giu lai.

In [ ]:
ev = pipeline.EvidenceBuilder()

## 5. VLM

Nap Qwen2.5-VL 7B. Cell nay lau nhat (~vai phut) va ton VRAM nhat.

`do_sample=False` (greedy) de ket qua on dinh giua cac lan chay - dieu kien de
so sanh prompt cho co nghia.

In [ ]:
vlm = pipeline.VLMRunner()

## 6. Chay Layer 1 tren val_binary, tach TP/FP

Theo Paper 4.5: chay gate tren val_binary, giu lai cac dong NO DA BAO DONG,
roi tach theo ground truth:

- `tp` - Label = 1, gate bao dung
- `fp` - Label = 0, gate bao nham

In [ ]:
alerts, feats = pipeline.load_alert_pool(l1)
print(f"\n{len(feats)} feature")
alerts["_set_type"].value_counts()

## 7. Lay mau — 10 alert moi loai tan cong

Chi lay nhom **tp** (alert dung). Nhanh danh gia false positive da bi bo: RAG
truy hoi bang chinh vector SHAP cua Layer 1 nen khong dung no de phu dinh
Layer 1 duoc (do thuc te: ca 5 FP deu khop ATTACK profile voi sim = 1.000).

10/nhan la muc cao nhat ma moi nhan deu dat (FTP_BruteForce chi co 10 alert)
-> du 14/14 nhan, tong 140 mau. Doi `per_class=2` de chay thu nhanh truoc.

In [ ]:
sample = pipeline.sample_stratified_tp(alerts, per_class=config.PER_CLASS)
sample["Label_Text"].value_counts()

## 8. MOT sample -> MOT report

Chay thu mot dong va in ca bao cao. Xem ky output nay truoc khi chay ca lo:
neu schema sai hoac anh hong thi sua o day, khong phai sau 100 luot.

`keep_images=True` de giu lai 5 anh ma xem.

In [ ]:
row = sample.iloc[0]

rec = pipeline.analyze_sample(
    row=row, l1=l1, ev=ev, vlm=vlm, feature_names=feats,
    prompt_variant="neutral",
    keep_images=True,
    meta={"row_idx": int(row["_row_idx"]),
          "set_type": row["_set_type"],
          "label": int(row["Label"]),
          "Label_Text": row["Label_Text"]},
)

print("run_error   :", rec.get("run_error"))
print("SU THAT     :", rec.get("Label_Text"), "| L1 proba", round(rec.get("xgb_proba", 0), 4))
print("RAG doan    :", rec.get("rag_predicted_label"),
      f"(sim={rec.get('rag_top_sim', 0):.3f}, zero_day={rec.get('rag_is_potential_zero_day')})")
print("VLM doan    :", rec.get("predicted_class"), "<-- M1 cham diem cai nay")
print("-" * 70)
print("severity    :", rec.get("severity"), "| MITRE:", rec.get("mitre_id"))
print("top_indicat :", rec.get("top_indicators"))
print("SHAP top-5  :", [f["name"] for f in (rec.get("shap_top_features") or [])])
print("conf logprob:", rec.get("confidence_logprob"),
      f"(token do: {rec.get('conf_token')!r})", "| tu khai:", rec.get("confidence_stated"))
print("n_sections  :", rec.get("n_sections"), "/6 | parse_error:", rec.get("parse_error"))
print("latency     :", round(rec.get("latency_s", 0), 1), "s")

In [ ]:
# Bao cao day du
print(rec.get("raw_output") or rec.get("traceback"))

In [ ]:
# Khoi RAG context da dua vao prompt
print(rec.get("rag_context"))

## 9. Chay ca lo

Ghi tung dong ngay khi xong (`append_jsonl`) - dut giua chung khong mat phan
da chay. Chay lai cell nay se GHI TIEP vao file cu; xoa file truoc neu muon
lam lai tu dau.

In [ ]:
PROMPT_VARIANT = "neutral"
OUT = os.path.join(config.OUT_DIR, f"results_{config.VARIANT}_{PROMPT_VARIANT}.jsonl")

# Bo qua cac sample da chay xong (cho phep chay tiep sau khi dut)
done = set()
if os.path.exists(OUT):
    with open(OUT, encoding="utf-8") as f:
        for ln in f:
            try:
                r = json.loads(ln)
                done.add((r.get("set_type"), r.get("row_idx")))
            except Exception:
                pass
    print(f"da co {len(done)} ban ghi - se bo qua")

t_start = time.time()
for i, (_, row) in enumerate(sample.iterrows(), 1):
    key = (row["_set_type"], int(row["_row_idx"]))
    if key in done:
        continue

    rec = pipeline.analyze_sample(
        row=row, l1=l1, ev=ev, vlm=vlm, feature_names=feats,
        prompt_variant=PROMPT_VARIANT,
        keep_images=False,
        meta={"row_idx": int(row["_row_idx"]),
              "set_type": row["_set_type"],
              "label": int(row["Label"]),
              "Label_Text": row["Label_Text"],
              "variant": config.VARIANT},
    )
    pipeline.append_jsonl(rec, OUT)

    el = time.time() - t_start
    print(f"[{i:3d}/{len(sample)}] {rec['set_type']:>2} "
          f"{str(rec.get('Label_Text'))[:20]:<20} -> {str(rec.get('predicted_class'))[:20]:<20} "
          f"{rec.get('latency_s', 0):5.1f}s  (da chay {el/60:.1f} phut)")

print(f"\nXong trong {(time.time()-t_start)/60:.1f} phut -> {OUT}")

## 10. Cham diem — G1 den G4

Xem METRIC.md de biet dinh nghia day du. Tom tat:

| | Do gi |
|---|---|
| **G1** M1 accuracy, M2 per-class, M3 MITRE | VLM co xac dinh dung loai tan cong? |
| **G2** M4 adherence, M5 harm, **M6 VLM vs RAG**, M7 schema | Dua bang chung hay tu bia? |
| **G3** M8 grounding, M9 halluc, M10 convergence | Report dung duoc? |
| **G4** M11 latency | Real-time? |

**M6 la chi so ban chinh** — so accuracy cua VLM voi accuracy cua chinh RAG:
VLM cong them, chep lai, hay lam hong ket qua truy hoi.

In [ ]:
df = metrics.load_results(OUT)
metrics.report_all(df)

## 11. Tung nhom rieng (neu can xem ky)

In [ ]:
m1 = metrics.attack_id_metrics(df)
pc = metrics.per_class(df)
display(pc)

In [ ]:
g2 = metrics.grounding_metrics(df)
g3 = metrics.report_quality(df)

## 12. Confidence

So hai nguon: `confidence_logprob` (xac suat token) va `confidence_stated`
(so model tu khai). Run cu cho thay cot tu khai gan nhu hang so 0.9 nen khong
quet nguong duoc — kiem tra lai o day.

In [ ]:
for col in ["confidence_logprob", "confidence_stated"]:
    metrics.confidence_auroc(df, col)
    sw = metrics.confidence_sweep(df, col)
    if len(sw):
        display(sw)

## 13. Latency + chat luong output

In [ ]:
display(metrics.latency_breakdown(df))

print("predicted_class:", df["predicted_class"].value_counts(dropna=False).head(10).to_dict())
print("parse_error    :", df["parse_error"].notna().sum(), "/", len(df))
print("run_error      :", df["run_error"].notna().sum(), "/", len(df))
print("zero-day flag  :", df["rag_is_potential_zero_day"].sum(), "/", len(df))

In [ ]:
# Ca VLM doan SAI - doc de hieu kieu loi
d = metrics.usable(df)
bad = d[~d.apply(lambda r: metrics._match(r["predicted_class"], r.get("Label_Text")), axis=1)]
print(f"{len(bad)}/{len(d)} ca sai")
display(bad[["Label_Text", "rag_predicted_label", "predicted_class",
             "rag_top_sim", "confidence_logprob"]].head(20))

## 14. Do do nhay voi prompt (tuy chon)

Chay cung mot bo mau qua 3 prompt variant de tach: ket qua den tu **bang chung**
hay tu **cach dat cau hoi**?

- `neutral` — khong uu tien nguon nao
- `trust_rag` — noi RAG dang tin
- `trust_visual` — noi RAG co the sai, doc chart truoc

Run cu (v6/v7/v13, cung 40 mau) cho 37/40 doi ket qua chi vi doi chu trong
prompt. Neu con lap lai o day thi do la ket qua phai bao cao.

Ton them 1 luot VLM moi sample moi variant.

In [ ]:
RUN_SENSITIVITY = False   # dat True de chay

if RUN_SENSITIVITY:
    sub = pipeline.sample_stratified_tp(alerts, per_class=3)
    OUT_S = os.path.join(config.OUT_DIR, f"results_sensitivity_{config.VARIANT}.jsonl")

    for variant in ["neutral", "trust_rag", "trust_visual"]:
        print(f"\n--- {variant} ---")
        for _, row in sub.iterrows():
            rec = pipeline.analyze_sample(
                row=row, l1=l1, ev=ev, vlm=vlm, feature_names=feats,
                prompt_variant=variant, keep_images=False,
                meta={"row_idx": int(row["_row_idx"]),
                      "set_type": row["_set_type"],
                      "label": int(row["Label"]),
                      "Label_Text": row["Label_Text"]},
            )
            pipeline.append_jsonl(rec, OUT_S)
            print(f"  {row['Label_Text']:<24} -> {rec.get('predicted_class')}")

    dfs = metrics.load_results(OUT_S)
    per, piv = metrics.prompt_sensitivity(dfs)
    display(per); display(piv)